# 🧠 Aula 2 — Minha primeira CNN (CIFAR-10)
Rode **uma célula por vez, em ordem** (botão ▶ ou Shift+Enter).
Se algo quebrar: *Ambiente de execução → Reiniciar e executar tudo*.

**Dica:** ative a GPU em *Ambiente de execução → Alterar tipo de ambiente de execução → GPU*. Sem GPU, use `epochs=5`.

## Passo 1 — Importar as bibliotecas
É como abrir a caixa de ferramentas antes de trabalhar.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from tensorflow import keras
from tensorflow.keras import layers

print("Keras versão:", keras.__version__)

## Passo 2 — Carregar os dados
CIFAR-10: 60 mil fotos coloridas de 32x32 pixels, em 10 categorias (50 mil para treino, 10 mil para teste).

In [ ]:
(x_train, y_train), (x_test, y_test) = keras.datasets.cifar10.load_data()

nomes = ["avião", "automóvel", "pássaro", "gato", "veado",
         "cachorro", "sapo", "cavalo", "navio", "caminhão"]

print("Treino:", x_train.shape, "| Teste:", x_test.shape)

**Checagem:** deve aparecer `(50000, 32, 32, 3)` e `(10000, 32, 32, 3)`.

## Passo 3 — Olhar as imagens
Sempre olhe seus dados antes de treinar.

In [ ]:
plt.figure(figsize=(8, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_train[i])
    plt.title(nomes[y_train[i][0]])
    plt.axis("off")
plt.show()

## Passo 4 — Normalizar
Pixels vão de 0 a 255. Dividimos por 255 para ficarem entre 0 e 1 (a rede aprende melhor). **Rode esta célula uma única vez!**

In [ ]:
x_train = x_train / 255.0
x_test = x_test / 255.0
print("Menor:", x_train.min(), "| Maior:", x_train.max())

## Passo 5 — Montar a CNN
Dois blocos **Conv + Pool**, depois Flatten e camadas densas.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Flatten(),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])

## Passo 6 — Ler o resumo do modelo

In [ ]:
model.summary()

## Passo 7 — Compilar
Definir o técnico (optimizer), o placar de erro (loss) e a métrica.

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

## Passo 8 — Treinar
Pode levar alguns minutos.

In [ ]:
history = model.fit(
    x_train, y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.1,
)

## Passo 9 e 10 — Curvas de aprendizado
As duas linhas sobem juntas? A de validação se afastou da de treino? (Se sim: *overfitting*.)

In [ ]:
plt.plot(history.history["accuracy"], label="treino")
plt.plot(history.history["val_accuracy"], label="validação")
plt.xlabel("época")
plt.ylabel("acurácia")
plt.legend()
plt.show()

## Passo 11 — A prova final (teste)
Anote seu número. Esperado: cerca de 0,65 a 0,72. Chute aleatório = 0,10.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=2)
print("Acurácia no teste:", test_acc)

## Passo 12 — Ver os palpites da rede
Onde ela errou? Faz sentido?

In [ ]:
probs = model.predict(x_test[:9])
palpites = np.argmax(probs, axis=1)

plt.figure(figsize=(8, 8))
for i in range(9):
    plt.subplot(3, 3, i + 1)
    plt.imshow(x_test[i])
    certo = nomes[y_test[i][0]]
    palpite = nomes[palpites[i]]
    plt.title(f"{palpite} (real: {certo})", fontsize=9)
    plt.axis("off")
plt.show()

## 🎯 Desafios (mude **uma coisa por vez** e anote o resultado)
1. Troque os filtros 32 e 64 por 64 e 128.
2. Adicione um terceiro bloco Conv.
3. Use Dropout para combater o overfitting (veja a célula abaixo).

Depois de reconstruir o modelo, rode de novo **compile** e **fit**.

In [ ]:
model = keras.Sequential([
    layers.Input(shape=(32, 32, 3)),
    layers.Conv2D(32, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(64, 3, activation="relu"),
    layers.MaxPooling2D(),
    layers.Conv2D(128, 3, activation="relu"),
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(64, activation="relu"),
    layers.Dense(10, activation="softmax"),
])
model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
history = model.fit(x_train, y_train, epochs=10,
                    batch_size=64, validation_split=0.1)
print("Teste:", model.evaluate(x_test, y_test, verbose=0)[1])